<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">

LING 593A-011<br>
Fall 2025<br>
Davo Acevedo-Cardona

# Company PAC Superset Construction Project Overview

## Task:
Integrate FEC political committee data with Snowflake contribution data to classify companies based on PAC activity and donation behavior.

---

## Dataset:
- FEC Committee Master (.txt files across election cycles)
- Snowflake PAC contribution dataset
- Company universe dataset
- Company–ticker crosswalk dataset

---

## What This Pipeline Does?

**Load:**  
Reads raw FEC committee master files and Snowflake PAC activity data.

**Filter:**  
Identifies corporate PACs by:
- committee type
- naming patterns (e.g., INC, CORP, PAC)

**Normalize:**  
Cleans committee names and company names by:
- lowercasing
- removing punctuation
- standardizing suffixes

**Merge:**  
Combines:
- FEC PAC data
- Snowflake contribution data

**Classify:**  
Determines PAC status for each company:
- has PAC
- has donations
- no donations

**Aggregate:**  
Rolls up PAC activity to the company level.

---

## Libraries Used

**Packages:**
- pandas  
- numpy  
- re  

---

## Pipeline

### Step 1: FEC Data Loading
- Load committee master files (multiple election cycles)  
- Assign column names  
- Combine into a unified dataset  

---

### Step 2: Corporate PAC Filtering
- Filter committees relevant to corporations  
- Remove irrelevant political committees  

---

### Step 3: Snowflake Data Integration
- Load Snowflake PAC activity dataset  
- Identify PACs with contribution records  

---

### Step 4: Name Normalization
- Clean and standardize committee names  
- Align naming across FEC and Snowflake datasets  

---

### Step 5: Matching & Merging
- Merge FEC PACs with Snowflake contribution data  
- Identify matched and unmatched PACs  

---

### Step 6: Donation Flagging
- Create flags:
  - `has_pac`
  - `has_pac_donations`  

---

### Step 7: Company-Level Aggregation
- Group PAC data by company  
- Compute final PAC status  

---

### Step 8: Final Classification
Assign each company to one of:

- `NO_PAC`  
- `PAC_DONATED`  
- `PAC_NO_DONATIONS`  

---

## Outputs

### CSV Files:
- `fec_committee_master.csv` → unified FEC dataset  
- `fec_pac_master.csv` → filtered corporate PACs  
- `fec_pacs_with_donation_flag.csv` → PACs with donation info  
- `fec_pacs_no_donations.csv` → PACs without donations  
- `company_pac_donation_status.csv` → company-level PAC summary  
- `company_pac_superset_U.csv` → final integrated dataset  

---

## Goal

Create a unified dataset that links **companies to their political activity**, enabling:

- PAC activity classification  
- corporate political behavior analysis  
- integration with executive and company datasets  

---

# Imports

In [1]:
import pandas as pd
import re

# Import the FEC Data

In [2]:
fec = pd.read_csv(
    "fec_committee_master.txt",
    sep="|",
    header=None,
    dtype=str,
    encoding="latin1"  # important for FEC files
)

#Assing columns
fec.columns = [
    "CMTE_ID",
    "CMTE_NM",
    "TRES_NM",
    "CMTE_ST1",
    "CMTE_ST2",
    "CMTE_CITY",
    "CMTE_ST",
    "CMTE_ZIP",
    "CMTE_DSGN",
    "CMTE_TP",
    "CMTE_PTY_AFFILIATION",
    "CMTE_FILING_FREQ",
    "ORG_TP",
    "CONNECTED_ORG_NM",
    "CAND_ID"
]

#Save the new csv
fec.to_csv("fec_committee_master.csv", index=False)
fec.head()

,CMTE_ID,CMTE_NM,TRES_NM,CMTE_ST1,CMTE_ST2,CMTE_CITY,CMTE_ST,CMTE_ZIP,CMTE_DSGN,CMTE_TP,CMTE_PTY_AFFILIATION,CMTE_FILING_FREQ,ORG_TP,CONNECTED_ORG_NM,CAND_ID
0,C00000059,HALLMARK CARDS PAC,SARAH MOE,2501 MCGEE,MD #500,KANSAS CITY,MO,64108,U,Q,UNK,M,C,NaN,NaN
1,C00000422,AMERICAN MEDICAL ASSOCIATION POLITICAL ACTION ...,"WALKER, KEVIN MR.","25 MASSACHUSETTS AVE, NW",SUITE 600,WASHINGTON,DC,200017400,B,Q,NaN,M,NaN,DELAWARE MEDICAL PAC,NaN
2,C00000489,D R I V E POLITICAL FUND CHAPTER 886,JERRY SIMS JR,3528 W RENO,NaN,OKLAHOMA CITY,OK,73107,U,N,NaN,Q,L,NaN,NaN
3,C00000547,KANSAS MEDICAL SOCIETY POLITICAL ACTION COMMITTEE,JERRY SLAUGHTER,623 SW 10TH AVE,NaN,TOPEKA,KS,666121627,U,Q,UNK,Q,M,KANSAS MEDICAL SOCIETY,NaN
4,C00000638,INDIANA STATE MEDICAL ASSOCIATION POLITICAL AC...,"ACHENBACH, GRANT MR.","322 CANAL WALK, CANAL LEVEL",NaN,INDIANAPOLIS,IN,46202,U,Q,NaN,T,M,NaN,NaN


# Filter PAC with connected ORGS

In [3]:
fec_pac_master = fec[
    fec["CONNECTED_ORG_NM"].notna() &
    (fec["CMTE_TP"] == "Q") &
    (fec["ORG_TP"] == "C")
]

fec_pac_master.shape

(829, 15)

In [4]:
fec_pac_master["CMTE_TP"].value_counts()
fec_pac_master["ORG_TP"].value_counts()
fec_pac_master["CONNECTED_ORG_NM"].head(10)


29                                      OLIN CORPORATION
65                  ELECTRIC COOPERATIVES OF MISSISSIPPI
79                        TEXAS INSTRUMENTS INCORPORATED
88                                  WEYERHAEUSER COMPANY
92                             FIRST HORIZON CORPORATION
97                                        CITIGROUP INC.
103                         NORFOLK SOUTHERN CORPORATION
106                         TRUIST FINANCIAL CORPORATION
119                            UNION PACIFIC CORPORATION
120    MEREDITH CORPORATION EMPLOYEES FUND FOR BETTER...
Name: CONNECTED_ORG_NM, dtype: object

# Create a clean P-set file

In [5]:
fec_p = fec_pac_master[["CMTE_ID", "CMTE_NM", "CONNECTED_ORG_NM", "CMTE_TP", "ORG_TP"]].copy()
fec_p.to_csv("fec_pac_master.csv", index=False)

# Upload Snowflake dats

In [6]:
def decode_b64url(data: str) -> str:
    if not data:
        return ""
    pad = len(data) % 4
    if pad:
        data += "=" * (4 - pad)
    return base64.urlsafe_b64decode(data).decode("utf-8", errors="replace")


def extract_plaintext(payload: dict) -> str:
    def walk(p):
        parts = [p]
        for c in p.get("parts", []) or []:
            parts.extend(walk(c))
        return parts

    for part in walk(payload):
        if part.get("mimeType") == "text/plain":
            return decode_b64url(part.get("body", {}).get("data", ""))

    return decode_b64url(payload.get("body", {}).get("data", ""))

# Brand Extraction

In [7]:
sf = pd.read_csv("snowflake_pac_activity.csv", dtype=str)

# Normalization

In [8]:
def norm_committee_name(s: str) -> str:
    if s is None:
        return ""
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)  # remove punctuation
    s = re.sub(r"\b(pac|political|action|committee|fund|the|and|of|for|employees)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

fec_p["committee_name_clean"] = fec_p["CMTE_NM"].map(norm_committee_name)
sf["committee_name_clean"] = sf["COMMITTEE_NAME"].map(norm_committee_name)

# Join and flag dontations

In [9]:
sf["TOTAL_AMOUNT"] = pd.to_numeric(sf["TOTAL_AMOUNT"], errors="coerce").fillna(0)

# Left join FEC to Snowflake Data

In [10]:
merged = fec_p.merge(
    sf[["committee_name_clean", "TOTAL_AMOUNT", "HAS_DONATED", "FIRST_CYCLE", "LAST_CYCLE"]],
    on="committee_name_clean",
    how="left",
    suffixes=("_fec", "_sf")
)

merged["TOTAL_AMOUNT"] = merged["TOTAL_AMOUNT"].fillna(0)
merged["HAS_DONATED"] = merged["HAS_DONATED"].fillna(False)

# Output

In [11]:
merged.to_csv("fec_pacs_with_donation_flag.csv", index=False)
no_donations = merged[merged["TOTAL_AMOUNT"] == 0]
no_donations.to_csv("fec_pacs_no_donations.csv", index=False)

In [12]:
print("Total FEC corporate PACs:", len(merged))
print("Matched to Snowflake PACs:", (merged["TOTAL_AMOUNT"] > 0).sum())
print("No donation activity:", (merged["TOTAL_AMOUNT"] == 0).sum())

Total FEC corporate PACs: 829
Matched to Snowflake PACs: 276
No donation activity: 553
